# Chapitre 9 — Restructurer les données

## Le problème

Les mêmes informations peuvent être rangées de plusieurs façons. Un tableau de ventes par produit et par mois peut avoir une colonne par mois (**format large**, agréable à lire) ou une ligne par couple produit-mois (**format long**, facile à filtrer, grouper, tracer).

```
 format large                              format long
 ┌──────────┬──────┬──────┬──────┐         ┌──────────┬──────┬────────┐
 │ produit  │ jan  │ fev  │ mar  │         │ produit  │ mois │ ventes │
 ├──────────┼──────┼──────┼──────┤   melt  ├──────────┼──────┼────────┤
 │ P01      │  10  │  12  │   9  │  ────►  │ P01      │ jan  │   10   │
 │ P02      │   4  │   0  │   7  │  ◄────  │ P01      │ fev  │   12   │
 └──────────┴──────┴──────┴──────┘  pivot  │ P01      │ mar  │    9   │
                                           │ P02      │ jan  │    4   │
                                           │ ...      │ ...  │  ...   │
                                           └──────────┴──────┴────────┘
```

D'autres situations demandent de changer la forme : une cellule qui contient une liste (réponse d'API), un JSON imbriqué, une variable catégorielle à transformer en colonnes numériques pour un modèle. Ce chapitre couvre ces transformations.

In [ ]:
import numpy as np
import pandas as pd

large = pd.DataFrame({"produit": ["P01", "P02", "P03"], "jan": [10, 4, 0], "fev": [12, 0, 3], "mar": [9, 7, 0]})
large

## 1. Du large au long : `melt`

In [ ]:
long = large.melt(id_vars="produit", var_name="mois", value_name="ventes")
long

- `id_vars` : les colonnes qui identifient une observation et restent telles quelles ;
- `value_vars` : les colonnes à « faire fondre » (par défaut, toutes les autres) ;
- `var_name`, `value_name` : les noms des deux nouvelles colonnes.

Une fois au format long, les opérations du chapitre 7 s'appliquent directement :

In [ ]:
long[long["ventes"] != 0].groupby("mois")["ventes"].sum()

## 2. Du long au large : `pivot` et `pivot_table`

`pivot` réorganise sans agréger. Il exige qu'une seule valeur corresponde à chaque couple (ligne, colonne).

In [ ]:
long.pivot(index="produit", columns="mois", values="ventes")

> **Comparaison.** `pivot` et `pivot_table` semblent faire la même chose. Que se passe-t-il avec `pivot` si un couple (produit, mois) apparaît deux fois ?

In [ ]:
doublon = pd.concat([long, pd.DataFrame({"produit": ["P01"], "mois": ["jan"], "ventes": [5]})])
doublon.pivot(index="produit", columns="mois", values="ventes")

In [ ]:
doublon.pivot_table(index="produit", columns="mois", values="ventes", aggfunc="sum")

`pivot` refuse d'arbitrer entre deux valeurs. `pivot_table` les agrège avec la fonction choisie. Règle pratique : `pivot` pour remettre en forme des données sans doublon, `pivot_table` dès qu'il faut agréger.

## 3. `stack` et `unstack` : déplacer un niveau d'index

Un `groupby` sur deux clés produit un MultiIndex. `unstack` fait passer le niveau intérieur en colonnes ; `stack` fait l'inverse.

In [ ]:
commandes = pd.read_csv("../donnees/demo/commandes.csv")
par_canal_statut = commandes.groupby(["canal", "statut"]).size()
par_canal_statut

In [ ]:
tableau = par_canal_statut.unstack()
tableau

In [ ]:
par_canal_statut.unstack(fill_value=0)

In [ ]:
tableau.stack()

`unstack(level=0)` déplace le premier niveau au lieu du dernier. `groupby(...).size().unstack()` est une alternative fréquente à `pivot_table(..., aggfunc="count")`.

## 4. Des listes dans les cellules : `explode`

Une API renvoie souvent des listes : pour chaque ville, la liste des heures et la liste des températures.

In [ ]:
meteo = pd.DataFrame({
    "ville": ["Bruxelles", "Liège"],
    "heure": [["00:00", "01:00", "02:00"], ["00:00", "01:00", "02:00"]],
    "temperature": [[6.1, 5.8, 5.5], [4.9, 4.6, 4.2]],
})
meteo

`explode` crée une ligne par élément de la liste, en répétant les autres colonnes.

> **Prédiction.** On veut une ligne par ville et par heure, soit 6 lignes. Combien de lignes produira chacune des deux écritures suivantes ?

In [ ]:
sequentiel = meteo.explode("heure").explode("temperature")
conjoint = meteo.explode(["heure", "temperature"])
len(sequentiel), len(conjoint)

In [ ]:
sequentiel.head(6)

L'écriture séquentielle éclate d'abord les heures (6 lignes), puis, **sur chacune de ces lignes**, la liste complète des températures : chaque heure est associée à chaque température, soit un produit cartésien de 3 × 3 par ville. Les listes qui vont ensemble doivent être éclatées **en une seule fois**, en passant la liste des colonnes.

In [ ]:
conjoint

Après `explode`, le type des colonnes est `object` : on le corrige avec `astype` ou `pd.to_numeric`.

In [ ]:
conjoint.dtypes

## 5. JSON imbriqué : `json_normalize`

Une réponse d'API contient des objets dans des objets. `pd.json_normalize` les aplatit en colonnes, en joignant les noms par des points.

In [ ]:
reponse = [
    {"id": 1, "client": {"nom": "Amina", "adresse": {"ville": "Anvers", "cp": "2000"}}, "total": 79.0},
    {"id": 2, "client": {"nom": "Lucas", "adresse": {"ville": "Liège", "cp": "4000"}}, "total": 549.0},
]
pd.json_normalize(reponse)

Quand chaque enregistrement contient une liste d'objets, `record_path` indique la liste à déplier et `meta` les champs à recopier sur chaque ligne :

In [ ]:
factures = [
    {"facture": "F1", "client": "Amina", "lignes": [{"produit": "P01", "qte": 1}, {"produit": "P06", "qte": 3}]},
    {"facture": "F2", "client": "Lucas", "lignes": [{"produit": "P09", "qte": 1}]},
]
pd.json_normalize(factures, record_path="lignes", meta=["facture", "client"])

## 6. Encoder des catégories pour un modèle : `get_dummies`

Un modèle numérique ne comprend pas `"web"` ou `"magasin"`. L'encodage *one-hot* crée une colonne par modalité, qui vaut 1 (ou `True`) quand la ligne a cette modalité.

In [ ]:
exemple = commandes[["commande_id", "canal", "quantite"]].head(5)
pd.get_dummies(exemple, columns=["canal"])

Paramètres utiles :

- `columns=` : les colonnes à encoder (par défaut, toutes les colonnes texte et catégorie) ;
- `prefix=` : le préfixe des nouvelles colonnes ;
- `drop_first=True` : supprime la première modalité, redondante (si ni `magasin` ni `telephone`, alors `web`), ce qui compte pour certains modèles linéaires ;
- `dtype=int` : des 0/1 au lieu de booléens.

In [ ]:
pd.get_dummies(exemple, columns=["canal"], drop_first=True, dtype=int)

### Le piège des données d'entraînement et de test

> **Pourquoi ?** On encode séparément un jeu d'entraînement et un jeu de test. Les deux tables obtenues ont-elles les mêmes colonnes ?

In [ ]:
train = pd.DataFrame({"canal": ["web", "magasin", "telephone", "web"]})
test = pd.DataFrame({"canal": ["web", "web"]})
pd.get_dummies(train).columns.tolist(), pd.get_dummies(test).columns.tolist()

Le jeu de test n'a qu'une modalité : il n'a qu'une colonne. Un modèle entraîné sur trois colonnes ne peut pas l'utiliser. Deux solutions :

- déclarer les catégories une fois pour toutes avec un type `category` commun ;
- réaligner les colonnes du test sur celles de l'entraînement avec `reindex`.

In [ ]:
categories = pd.CategoricalDtype(["magasin", "telephone", "web"])
pd.get_dummies(test.astype({"canal": categories})).columns.tolist()

In [ ]:
cols_train = pd.get_dummies(train).columns
pd.get_dummies(test).reindex(columns=cols_train, fill_value=False)

Dans un projet réel, on confie souvent cette étape à un encodeur de scikit-learn (`OneHotEncoder`), qui mémorise les catégories vues à l'entraînement. Le principe est le même.

## 7. Discrétiser une variable numérique : `cut` et `qcut`

Transformer un âge ou un prix en tranches est une autre restructuration fréquente.

In [ ]:
clients = pd.read_csv("../donnees/demo/clients.csv")
clients["tranche_age"] = pd.cut(clients["age"], bins=[18, 30, 45, 60, 100], labels=["18-30", "31-45", "46-60", "60+"])
clients["tranche_age"].value_counts().sort_index()

In [ ]:
pd.qcut(clients["age"], q=4).value_counts().sort_index()      # quatre tranches d'effectifs égaux

`cut` découpe selon des bornes fixées ; `qcut` selon les quantiles, pour obtenir des groupes de même taille. Les valeurs manquantes restent manquantes.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| `explode` successifs sur des listes parallèles | produit cartésien, lignes multipliées | `explode([col1, col2])` |
| `pivot` avec des couples en double | `ValueError: Index contains duplicate entries` | `pivot_table` avec `aggfunc` |
| `get_dummies` sur train et test séparément | colonnes différentes | catégories communes ou `reindex` |
| Colonnes `object` après `explode` | calculs impossibles ou lents | `astype` / `to_numeric` |
| `unstack` avec des combinaisons absentes | `NaN` | `fill_value=0` |

## Exercices

- Exercice supplémentaire **S9** : `get_dummies` sur des données d'entraînement et de test.
- L'exercice Bstorm **API météo** (chapitre 11) utilise `json_normalize` et `explode`.

## À retenir

- Format long pour calculer, format large pour lire.
- `melt` : large → long ; `pivot` / `pivot_table` : long → large ; `stack` / `unstack` : index ↔ colonnes.
- `explode` sur plusieurs colonnes à la fois pour des listes parallèles.
- `json_normalize` aplatit le JSON imbriqué (`record_path`, `meta`).
- `get_dummies` encode les catégories ; mêmes colonnes en entraînement et en test.
- `cut` et `qcut` transforment un nombre en tranches.